# Contents:
* ECLAT
* Patterns Evaluation

## 1- ECLAT

In [ ]:
! pip install pyECLAT

In [ ]:
from pyECLAT import Example1, Example2
ex1 = Example1().get()
ex1

,0,1,2,3
0,milk,beer,bread,butter
1,coffe,bread,butter,NaN
2,coffe,bread,butter,NaN
3,milk,coffe,bread,butter
4,beer,NaN,NaN,NaN
5,butter,NaN,NaN,NaN
6,bread,NaN,NaN,NaN
7,bean,NaN,NaN,NaN
8,rice,bean,NaN,NaN
9,rice,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame([
    ["Bread",  "Butter", "Jam",   np.nan],
    ["Butter", "Coke",   np.nan,  np.nan],
    ["Butter", "Milk",   np.nan,  np.nan],
    ["Bread",  "Butter", "Coke",  np.nan],
    ["Bread",  "Milk",   np.nan,  np.nan],
    ["Butter", "Milk",   np.nan,  np.nan],
    ["Bread",  "Milk",   np.nan,  np.nan],
    ["Bread",  "Butter", "Milk",  "Jam"],
    ["Bread",  "Butter", "Milk",  np.nan],
])

# IMPORTANT: columns must be numeric for ECLAT
df.columns = [0, 1, 2, 3]

df

,0,1,2,3
0,Bread,Butter,Jam,NaN
1,Butter,Coke,NaN,NaN
2,Butter,Milk,NaN,NaN
3,Bread,Butter,Coke,NaN
4,Bread,Milk,NaN,NaN
5,Butter,Milk,NaN,NaN
6,Bread,Milk,NaN,NaN
7,Bread,Butter,Milk,Jam
8,Bread,Butter,Milk,NaN


In [ ]:
from pyECLAT import ECLAT
eclat_instance = ECLAT(data=df, verbose=True) #verbose=True to see the loading bar
eclat_instance

100%|██████████| 5/5 [00:00<00:00, 694.26it/s]


In [ ]:
eclat_instance.df_bin   #generate a binary dataframe, that can be used for other analyzes.


,Jam,Coke,Butter,Milk,Bread
0,1,0,1,0,1
1,0,1,1,0,0
2,0,0,1,1,0
3,0,1,1,0,1
4,0,0,0,1,1
5,0,0,1,1,0
6,0,0,0,1,1
7,1,0,1,1,1
8,0,0,1,1,1


In [ ]:

eclat_instance.uniq_    #a list with all the names of the different items


['Jam', 'Coke', 'Butter', nan, 'Milk', 'Bread']

In [ ]:
get_ECLAT_indexes, get_ECLAT_supports = eclat_instance.fit(min_support=0.2,
                                                           min_combination=1,
                                                           max_combination=3,
                                                           separator=' & ',
                                                           verbose=True)

Combination 1 by 1


5it [00:00, 170.16it/s]


Combination 2 by 2


10it [00:00, 187.42it/s]


Combination 3 by 3


10it [00:00, 236.19it/s]


In [ ]:
get_ECLAT_indexes

{'Jam': [0, 7],
 'Coke': [1, 3],
 'Butter': [0, 1, 2, 3, 5, 7, 8],
 'Milk': [2, 4, 5, 6, 7, 8],
 'Bread': [0, 3, 4, 6, 7, 8],
 'Jam & Butter': [0, 7],
 'Jam & Bread': [0, 7],
 'Coke & Butter': [1, 3],
 'Butter & Milk': [2, 5, 7, 8],
 'Butter & Bread': [0, 3, 7, 8],
 'Milk & Bread': [4, 6, 7, 8],
 'Jam & Butter & Bread': [0, 7],
 'Butter & Milk & Bread': [7, 8]}

In [ ]:
get_ECLAT_supports

{'Jam': 0.2222222222222222,
 'Coke': 0.2222222222222222,
 'Butter': 0.7777777777777778,
 'Milk': 0.6666666666666666,
 'Bread': 0.6666666666666666,
 'Jam & Butter': 0.2222222222222222,
 'Jam & Bread': 0.2222222222222222,
 'Coke & Butter': 0.2222222222222222,
 'Butter & Milk': 0.4444444444444444,
 'Butter & Bread': 0.4444444444444444,
 'Milk & Bread': 0.4444444444444444,
 'Jam & Butter & Bread': 0.2222222222222222,
 'Butter & Milk & Bread': 0.2222222222222222}

## 2- Patterns Evaluation


In [ ]:
import pandas as pd

def measures_from_transactions(transactions, X, Y):
    """
    transactions: list of sets, e.g. [{"Bread","Butter"}, {"Bread"}, ...]
    X, Y: item names (strings)
    Returns a dict with:
      - counts
      - support
      - confidence (X->Y, Y->X)
      - lift (X->Y)
      - interest (same as lift via alternative formula)
      - contingency table counts (a,b,c,d)
      - h-confidence
    """
    N = len(transactions)

    def has(item, t):
        return item in t

    # count frequencies
    nX  = sum(has(X, t) for t in transactions)
    nY  = sum(has(Y, t) for t in transactions)
    nXY = sum(has(X, t) and has(Y, t) for t in transactions)

    # Contingency table:
    '''
     a c
     b d
    '''
    # a = #(X and Y)
    # b = #(X and not Y)
    # c = #(not X and Y)
    # d = #(not X and not Y)
    a = nXY
    b = nX - nXY
    c = nY - nXY
    d = N - (a + b + c)

    # Supports
    suppX  = nX / N
    suppY  = nY / N
    suppXY = nXY / N

    # Confidence
    conf_XY = (suppXY / suppX) if suppX > 0 else 0.0 # X -> Y
    conf_YX = (suppXY / suppY) if suppY > 0 else 0.0 # Y -> X

    # Lift / Interest
    lift_XY = (conf_XY / suppY) if suppY > 0 else 0.0
    interest_XY = (suppXY / (suppX * suppY)) if (suppX > 0 and suppY > 0) else 0.0

    # H-confidence
    h_conf = min(conf_XY, conf_YX)

    return {
        "N": N,
        "count_X": nX,
        "count_Y": nY,
        "count_XY": nXY,
        "support_X": suppX,
        "support_Y": suppY,
        "support_XY": suppXY,
        "conf_X->Y": conf_XY,
        "conf_Y->X": conf_YX,
        "lift_X->Y": lift_XY,
        "interest_X->Y": interest_XY,
        "contingency_a(X∧Y)": a,
        "contingency_b(X∧¬Y)": b,
        "contingency_c(¬X∧Y)": c,
        "contingency_d(¬X∧¬Y)": d,
        "h_conf(X,Y)": h_conf,
    }


def print_contingency_table(result, X, Y):
    a = result["contingency_a(X∧Y)"]
    b = result["contingency_b(X∧¬Y)"]
    c = result["contingency_c(¬X∧Y)"]
    d = result["contingency_d(¬X∧¬Y)"]
    N = result["N"]

    table = pd.DataFrame(
        [[a, b, a + b],
         [c, d, c + d],
         [a + c, b + d, N]],
        index=[X, f"¬{X}", "Total"],
        columns=[Y, f"¬{Y}", "Total"]
    )
    return table


# -------------------------
# Example: the simple dataset used earlier
# -------------------------
transactions = [
    {"Bread", "Butter"},  # T1
    {"Bread"},            # T2
    {"Butter"},           # T3
    {"Bread", "Butter"},  # T4
    {"Bread"},            # T5
]

X, Y = "Bread", "Butter"

res = measures_from_transactions(transactions, X, Y)


print("=== Measures ===")
for k, v in res.items():
    if isinstance(v, float):
        print(f"{k:>20}: {v:.4f}")
    else:
        print(f"{k:>20}: {v}")

print("\n=== Contingency Table ===")
print(print_contingency_table(res, X, Y))


=== Measures ===
                   N: 5
             count_X: 4
             count_Y: 3
            count_XY: 2
           support_X: 0.8000
           support_Y: 0.6000
          support_XY: 0.4000
           conf_X->Y: 0.5000
           conf_Y->X: 0.6667
           lift_X->Y: 0.8333
       interest_X->Y: 0.8333
  contingency_a(X∧Y): 2
 contingency_b(X∧¬Y): 2
 contingency_c(¬X∧Y): 1
contingency_d(¬X∧¬Y): 0
         h_conf(X,Y): 0.5000

=== Contingency Table ===
        Butter  ¬Butter  Total
Bread        2        2      4
¬Bread       1        0      1
Total        3        2      5
